In [1]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

resnet_model = ResNet50(input_shape=(224,224,3),include_top=False)
resnet_model.traiable = True

model = Sequential()
model.add(resnet_model)
model.add(Flatten())
model.add(Dense(3, activation='softmax')) # 3개 클래스 분류
model.summary()

I0000 00:00:1779583298.550289   37473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779583298.596376   37473 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779583299.525072   37473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1779583300.210025   37473 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries co

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │       301,059 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,888,771 (91.13 MB)

 Trainable params: 23,835,651 (90.93 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(rotation_range=20,
                               width_shift_range=0.2, height_shift_range=0.2,
                               horizontal_flip=True)
train_data = train_gen.flow_from_directory('./glaucoma/train', target_size=(224,224),
                                           batch_size=32, class_mode='sparse')

test_gen = ImageDataGenerator()
test_data = test_gen.flow_from_directory('./glaucoma/test', target_size=(224,224),
                                         batch_size=32, class_mode='sparse')

Found 1394 images belonging to 3 classes.
Found 150 images belonging to 3 classes.


In [3]:
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard

checkpoint = ModelCheckpoint("best_model-glaucoma.keras", save_best_only=True)
tensorboard = TensorBoard(log_dir="./logs-glaucoma", histogram_freq=1, embeddings_freq=1)

In [5]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
              metrics=['accuracy'])
model.fit(train_data, validation_data=test_data, epochs=20, callbacks=[checkpoint, tensorboard])

Epoch 1/20


I0000 00:00:1779583428.001256   37473 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1779583443.180388   37938 service.cc:153] XLA service 0x76d8a403fe10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779583443.180425   37938 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5090, Compute Capability 12.0a (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1779583443.692279   37938 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779583446.667895   37938 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1779583446.976244   37938 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_43286__.390
I0000 00:00:1779583459.131798   38929 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion

 1/44 ━━━━━━━━━━━━━━━━━━━━ 27:58 39s/step - accuracy: 0.3333 - loss: 1.9832

I0000 00:00:1779583470.101215   37939 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_43286__.390
I0000 00:00:1779583477.650813   40163 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_15', 28 bytes spill stores, 28 bytes spill loads

I0000 00:00:1779583478.345191   40252 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_36', 52 bytes spill stores, 52 bytes spill loads

I0000 00:00:1779583480.761458   40163 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_27', 52 bytes spill stores, 52 bytes spill loads



 2/44 ━━━━━━━━━━━━━━━━━━━━ 14:06 20s/step - accuracy: 0.3667 - loss: 5.5663

I0000 00:00:1779583486.994565   37939 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_36', 52 bytes spill stores, 52 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_27', 52 bytes spill stores, 52 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_15', 28 bytes spill stores, 28 bytes spill loads



44/44 ━━━━━━━━━━━━━━━━━━━━ 79s 924ms/step - accuracy: 0.6549 - loss: 2.3602 - val_accuracy: 0.5200 - val_loss: 173328.1562
Epoch 2/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 14s 311ms/step - accuracy: 0.7001 - loss: 2.0018 - val_accuracy: 0.5200 - val_loss: 84801304.0000
Epoch 3/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 15s 343ms/step - accuracy: 0.7152 - loss: 1.7055 - val_accuracy: 0.5200 - val_loss: 76572.3828
Epoch 4/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 15s 351ms/step - accuracy: 0.7174 - loss: 0.6196 - val_accuracy: 0.6467 - val_loss: 976.5575
Epoch 5/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 16s 353ms/step - accuracy: 0.7310 - loss: 0.5707 - val_accuracy: 0.6467 - val_loss: 4.6283
Epoch 6/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 15s 350ms/step - accuracy: 0.7432 - loss: 0.5604 - val_accuracy: 0.6800 - val_loss: 0.7540
Epoch 7/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 15s 345ms/step - accuracy: 0.7410 - loss: 0.5669 - val_accuracy: 0.7067 - val_loss: 0.6372
Epoch 8/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 15s 347ms/step - accuracy: 0.7461 - loss: 0.5532 - v

In [1]:
from tensorflow.keras.models import keras

best_model = load_model("best_model-glaucoma.keras") # best model load

ImportError: cannot import name 'keras' from 'tensorflow.keras.models' (/workspace/.venv/lib/python3.12/site-packages/keras/_tf_keras/keras/models/__init__.py)

In [ ]:
from tensorflow.keras.preprocessing import image
import numby as np 
img = image.load_img('test.png', target_size=(224,224))
x = image.img_to_array(img).reshape(-1, 224, 224, 3)
pred = model.predict(x)
print(pred) # test.png는 advanced glaucoma 중 하나

print(np.argmax(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
[[0. 0. 1.]]


 tensorboard --logdir=./logs-glaucoma/

In [ ]:
import numpy as np

# 1. 클래스 이름 정의 (본인의 데이터셋 순서에 맞게 이름을 수정하세요)
# 예: ['정상', '초기 녹내장', '말기 녹내장']
class_names = ['Advanced_glaucoma', 'Early_glucoma', 'Normal_control']

# 2. 가장 높은 확률을 가진 인덱스 추출 (결과는 2가 나옵니다)
predicted_index = np.argmax(pred[0])

# 3. 결과 출력
print(f"예측 결과: {class_names[predicted_index]} (확률: {pred[0][predicted_index] * 100}%)")